# PDF Extraction: Tokens & Cost

How many tokens does it cost to extract structured data from a PDF?
We compare **GitHub Models hotet gpt 4.1** (8k limit) vs **Anthropic sonnet 4.6** (200k context).

In [2]:
import base64
import os
import sys
from typing import List

import chatlas as ctl
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv()

True

## 1. The PDF: file size vs wire size

APIs can't receive raw binary — PDFs are sent as **base64**, which adds ~33% overhead.

In [ ]:
PDF_PATH = "eval.pdf"

raw_bytes = os.path.getsize(PDF_PATH)
b64_bytes = len(base64.b64encode(open(PDF_PATH, "rb").read()))

print(f"PDF on disk:       {raw_bytes:>10,} bytes  ({raw_bytes/1024:.1f} KB)")
print(f"Base64 on wire:    {b64_bytes:>10,} bytes  ({b64_bytes/1024:.1f} KB)")
print(f"Overhead:          {b64_bytes/raw_bytes:.0%}")

## 2. Token estimate *before* sending

`chat.token_count()` estimates how many tokens the input will consume.  
This lets you check whether you'll hit a provider's limit **before** spending money.

In [ ]:
pdf = ctl.content_pdf_file(PDF_PATH)


# -- Pydantic schema (same as 03-pdf.py) --
class LikertScaleRow(BaseModel):
    """Single row of likert scale results"""
    question_type: str = Field(description="General heading the questions are under")
    question: str = Field(description="Question that was asked in the survey")
    N: int = Field(description="Number of responses invited")
    n: int = Field(description="Number of responses")
    sd: int = Field(description="Number of strongly disagree")
    d: int = Field(description="Number of disagree")
    neutral: int = Field(description="Number of neutral")
    a: int = Field(description="Number of agree")
    sa: int = Field(description="Number of strongly agree")
    na: int = Field(description="Number of not applicable")
    im: float = Field(description="interpolated median value")
    pf: str = Field(description="percent favorable rating")
    di: float = Field(description="dispersion index")


class LikertScaleResults(BaseModel):
    """Collection of Likert scale results"""
    results: List[LikertScaleRow]

In [ ]:
SYSTEM_PROMPT = """
You will be given a PDF of student feedback results from a course.
There are multiple questions the students are asked and the likert
data is summarized across multiple tables.
Extract all the data in the tables.
"""

# --- GitHub Models estimate ---
ghm = ctl.ChatGithub(model="gpt-4.1-mini", system_prompt=SYSTEM_PROMPT)
ghm_tokens = ghm.token_count(pdf, data_model=LikertScaleResults)

# --- Anthropic estimate ---
claude = ctl.ChatAnthropic(system_prompt=SYSTEM_PROMPT)
claude_tokens = claude.token_count(pdf, data_model=LikertScaleResults)

print(f"Estimated input tokens:")
print(f"  GitHub Models (gpt-4.1-mini): {ghm_tokens:>8,} tokens")
print(f"  Anthropic (Claude Sonnet):    {claude_tokens:>8,} tokens")
print()
print(f"GitHub Models free-tier limit:       8,000 tokens")
print(f"Anthropic context window:          200,000 tokens")

## 3. Why GitHub Models fails

The PDF alone exceeds the 8,000-token input limit on GitHub Models' free tier.  
This is why `03-pdf.py` would either work with `ChatAnthropic()`, or with OpenAI: same model, different provider limitations.

In [1]:
try:
    ghm.chat_structured(pdf, data_model=LikertScaleResults, echo="none")
except Exception as e:
    print(f"GitHub Models error: {type(e).__name__}")
    print(f"  {e.message if hasattr(e, 'message') else e}")

GitHub Models error: NameError
  name 'ghm' is not defined


## 4. Run with Anthropic and calculate token usage & cost

In [ ]:
claude = ctl.ChatAnthropic(system_prompt=SYSTEM_PROMPT)

dat = claude.chat_structured(pdf, data_model=LikertScaleResults, echo="none")

results = pd.DataFrame([r.model_dump() for r in dat.results])
results

In [ ]:
# Actual tokens used (reported by the API)
tokens = claude.get_tokens()
print("Tokens per turn:")
for i, t in enumerate(tokens):
    print(f"  Turn {i}: {t}")

print(f"\nEstimated cost: ${claude.get_cost():.4f} USD")

## 5. Text vs binary: how providers "see" the PDF

Both providers receive the PDF as **base64-encoded binary** — the full file,  
not extracted text. The model's vision/document pipeline parses it internally.

| What's sent | Format | Size |
|-------------|--------|------|
| PDF file on disk | Binary | ~410 KB |
| PDF over the API | Base64 string | ~547 KB (+33%) |
| Token count | Provider-specific | See above |

You pay for the *full binary (and encoded) PDF*, not just the text in it.  
A 3-page PDF with charts and logos costs more tokens than 3 pages of plain text.

In [ ]:
# Compare: same content as plain text vs PDF
plain_text = results.to_string()

text_tokens = claude.token_count(plain_text, data_model=LikertScaleResults)
pdf_tokens = claude.token_count(pdf, data_model=LikertScaleResults)

print(f"Tokens for extracted table as plain text: {text_tokens:>8,}")
print(f"Tokens for original PDF:                  {pdf_tokens:>8,}")
print(f"PDF costs {pdf_tokens / text_tokens:.1f}x more tokens than plain text")

## Summary

| | GitHub Models (free) | Anthropic |
|---|---|---|
| Context window | 8,000 tokens | 200,000 tokens |
| PDF support | ✅ (format) | ✅ |
| This PDF fits? | ❌ | ✅ |
| Cost | Free | ~$0.01–0.05 per call |

**Takeaway:** Worth using `token_count()` and knowing your prices.